In [ ]:
# try:
#     from cellsweep import denoise_count_matrix
# except ImportError:
#     print("cellsweep not found, installing...")
#     !pip install -U -q cellsweep[analysis]

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import math
import itertools
import yaml
import requests
import matplotlib.pyplot as plt
import anndata as ad
import scipy.sparse as sp
from collections import OrderedDict
import seaborn as sns
import scanpy as sc
from cellsweep import denoise_count_matrix
import cellsweep.utils as cs_utils

cellsweep_dir = os.path.dirname(os.path.abspath(""))

# 8 cubed

In [ ]:
adata_raw_parent_dir = "/mnt/data1"
adata_filtered_dir = "/mnt/data1/8_cube_cellbender_raw"

verbose = 2  # 2 debug, 1 info, 0 warning, -1 error, -2 critical
overwrite = False  # overwrite existing files
threads = 32  # for cellsweep and CellBender (if use_cuda=False)

In [ ]:
dataset_name = "8cubed"

# create directories for data, output
data_dir = os.path.join(cellsweep_dir, "notebooks", "data", dataset_name)
os.makedirs(data_dir, exist_ok=True)

out_dir = os.path.join(cellsweep_dir, "notebooks", "output", dataset_name)
os.makedirs(out_dir, exist_ok=True)

eight_cubed_markers_path = os.path.join(data_dir, "8_cube_marker_genes.csv")
gene_id_name_map_path = os.path.join(data_dir, "gene_id_name_map.csv")

cellsweep_max_iter = 2000
cellsweep_init_beta = 0.1
cellsweep_init_alpha = 0.9

if not os.path.exists(adata_raw_parent_dir):
    raise ValueError(f"adata_raw_parent_dir {adata_raw_parent_dir} does not exist.")
if not os.path.exists(adata_filtered_dir):
    raise ValueError(f"adata_filtered_dir {adata_filtered_dir} does not exist.")
if not os.path.exists(gene_id_name_map_path):
    !wget -O {gene_id_name_map_path} https://caltech.box.com/shared/static/nagn3icgony8u8408cofqaf2rs8yivjk.csv

## For trem2, stop running here and skip to the end

In [ ]:
plate_to_tissues = {}
plates = ["igvf_003", "igvf_004", "igvf_005", "igvf_007", "igvf_008b", "igvf_009", "igvf_010", "igvf_011"]
for plate in plates:
    plate_dir = os.path.join(adata_raw_parent_dir, plate)
    plate_to_tissues[plate] = [tissue for tissue in os.listdir(plate_dir)]
        
plate_to_tissues

## Raw

In [ ]:
def load_8cubed_raw_data(adata_dir, backed=None, debug=False):
    # plate: adata
    try:
        from tqdm import tqdm
        tq = tqdm
    except ImportError:
        # no tqdm → identity function
        tq = lambda x, *args, **kwargs: x

    # plates = ["igvf_003", "igvf_004", "igvf_005", "igvf_007", "igvf_008b", "igvf_009", "igvf_010", "igvf_011"]

    adata_dict = {}
    for plate in tq(plates, desc="Loading AnnData plates"):
        adatas_plate = []
        plate_dir = os.path.join(adata_dir, plate)
        tissues = os.listdir(plate_dir)
        if len(tissues) < 2 or len(tissues) > 3:
            print(f"[WARN] Plate directory {plate_dir} does not contain 2-3 tissues, skipping")
            continue
        for tissue in tissues:
            adata_path = os.path.join(plate_dir, tissue, "adata.h5ad")
            adata = ad.read_h5ad(adata_path, backed=backed)
            if debug:
                adata = adata[:10_000, :].to_memory()
            adata.var_names_make_unique()
            # merge GonadsMale and GonadsFemale into Gonads
            adata.obs["Tissue_original"] = adata.obs["Tissue"]
            adata.obs["Tissue"] = adata.obs["Tissue"].replace({
                "GonadsMale": "Gonads",
                "GonadsFemale": "Gonads",
            })
            adatas_plate.append(adata)
        adata_plate = ad.concat(adatas_plate, axis=0, join="inner", index_unique=None)
        adata_dict[plate] = adata_plate
    return adata_dict

def load_8cubed_filtered_data(adata_dir, debug=False, obs_only=True, tissues_to_include=None):
    # tissue: adata
    try:
        from tqdm import tqdm
        tq = tqdm
    except ImportError:
        # no tqdm → identity function
        tq = lambda x, *args, **kwargs: x

    backed = "r" if obs_only else None

    # List .h5ad files
    files = sorted([os.path.join(adata_dir, f) for f in os.listdir(adata_dir) if f.endswith(".h5ad")])

    if not files:
        raise ValueError(f"No .h5ad files found in: {adata_dir}")

    print(f"Found {len(files)} files:")
    for f in files:
        print("  -", f)

    # Load them
    adata_obs_dict = {}
    for filename in tq(files, desc="Loading AnnData"):
        tissue = os.path.basename(filename).replace("_annotated.h5ad", "")
        if tissues_to_include is not None and tissue not in tissues_to_include:
            print(f"[INFO] Skipping tissue {tissue} not in include list")
            continue
        
        # Load metadata only
        adata = ad.read_h5ad(filename, backed=backed)
        obs = adata.obs.copy()   # copy small metadata only
        if obs_only:
            del adata                # drop the heavy object

        # Debug: limit rows
        if debug:
            obs = obs.iloc[:1_000].copy()

        # Tissue cleanup
        obs["Tissue_original"] = obs["Tissue"]
        obs["Tissue"] = obs["Tissue"].replace({
            "GonadsMale": "Gonads",
            "GonadsFemale": "Gonads",
        })

        # ensure correct index
        if "cellID" in obs.columns:
            obs = obs.set_index("cellID")

        if obs_only:
            adata_obs_dict[tissue] = obs
        else:
            adata.obs = obs
            if "cellbender_counts" in adata.layers:
                adata.X = adata.layers["cellbender_counts"]
            adata_obs_dict[tissue] = adata

    # --- Combine Gonads (obs-only) ---
    if "GonadsMale" in adata_obs_dict and "GonadsFemale" in adata_obs_dict:
        if obs_only:
            adata_obs_dict["Gonads"] = pd.concat(
                [adata_obs_dict["GonadsMale"], adata_obs_dict["GonadsFemale"]],
                axis=0,
                join="inner"
            )
        else:
            adata_obs_dict["Gonads"] = adata_obs_dict["GonadsMale"].concatenate(adata_obs_dict["GonadsFemale"])
        
        del adata_obs_dict["GonadsMale"]
        del adata_obs_dict["GonadsFemale"]

    return adata_obs_dict

def merge_filtered_celltype_and_cluster_into_raw(adata_raw_dict, adata_filtered_tissue_obs_dict):
    adata_raw_plate_with_celltype_dict = {}

    for plate, adata in adata_raw_dict.items():
        if "celltype" in adata.obs.columns:
            print(f"[WARN] Plate {plate} already has celltype column — skipping")
            adata_raw_plate_with_celltype_dict[plate] = adata
            continue

        # Make a copy so original isn't modified
        # adata = adata.copy()

        # Start by initializing the column with Empty Droplet everywhere
        adata.obs["celltype"] = "Empty Droplet"
        adata.obs["leiden"] = "unassigned"

        # Loop over tissues for this plate
        for tissue in adata.obs["Tissue"].unique():

            if tissue not in adata_filtered_tissue_obs_dict:
                print(f"[WARN] Tissue {tissue} not in filtered dict — skipping")
                continue

            adata_obs_filtered = adata_filtered_tissue_obs_dict[tissue]

            cols_to_merge = ["celltype", "leiden"]
            missing = [c for c in cols_to_merge if c not in adata_obs_filtered.columns]
            if missing:
                print(f"[WARN] Filtered {tissue} missing columns {missing} — skipping")
                continue

            # Filtered celltypes index
            filt_obs = adata_obs_filtered[cols_to_merge].copy()

            if "Subpool" in filt_obs.index[0]:
                filt_obs.index = filt_obs.index.str.replace("Subpool", "Sublibrary", regex=False)

            # Only cells that exist in this raw plate
            common_idx = adata.obs.index.intersection(filt_obs.index)

            if len(common_idx) == 0:
                print(f"[INFO] No shared cells between plate {plate} and {tissue}")
                continue

            # Insert celltypes for those cells
            adata.obs.loc[common_idx, cols_to_merge] = filt_obs.loc[common_idx, cols_to_merge]

        # Store final annotated plate
        adata_raw_plate_with_celltype_dict[plate] = adata

    return adata_raw_plate_with_celltype_dict

In [ ]:
adata_raw_dict = load_8cubed_raw_data(adata_raw_parent_dir)
adata_filtered_tissue_obs_dict = load_8cubed_filtered_data(adata_filtered_dir, obs_only=True)

## Knee plot - use this output to estimate umi_cutoff

In [ ]:
expected_cells = {plate: 0 for plate in adata_raw_dict.keys()}
for adata_obs in adata_filtered_tissue_obs_dict.values():
    for plate in adata_obs["plate"].unique():
        if plate in expected_cells:
            # sum value counts from adata
            expected_cells[plate] += int((adata_obs["plate"] == plate).sum())
expected_cells

In [ ]:
umi_cutoffs = {}
for plate, adata_raw in adata_raw_dict.items():
    print(f"Processing plate {plate} for knee plot with expected cells = {expected_cells[plate]}")
    umi_cutoff = cs_utils.knee_plot(adata_raw, expected_cells=expected_cells[plate], title=f"Knee Plot - {plate}", out_path=os.path.join(out_dir, f"plate_{plate}", f"knee_plot_{plate}.png"))
    umi_cutoffs[plate] = umi_cutoff

In [ ]:
#!!! optionally update umi_cutoffs from knee plot - required for None values
# umi_cutoffs = {
#     "igvf_003": None,
#     "igvf_004": None,
#     "igvf_005": None,
#     "igvf_007": None,
#     "igvf_008b": None,
#     "igvf_009": None,
#     "igvf_010": None,
#     "igvf_011": None,
# }

In [ ]:
adata_raw_dict = merge_filtered_celltype_and_cluster_into_raw(adata_raw_dict, adata_filtered_tissue_obs_dict)

## Organize cellbender adatas

In [ ]:
# plates = ["igvf_003", "igvf_004", "igvf_005", "igvf_007", "igvf_008b", "igvf_009", "igvf_010", "igvf_011"]  # ["igvf_011", "igvf_003", "igvf_004", "igvf_005"]  # ["igvf_007", "igvf_008b"]  # ["igvf_009", "igvf_010"]
tissues_to_include = None  # ["Gastrocnemius", "CortexHippocampus", "Heart", "Liver", "HypothalamusPituitary"]  # ["HypothalamusPituitary", "GonadsMale", "GonadsFemale", "Adrenal"]  # ["Adrenal", "Kidney", "Gastrocnemius"]
adata_filtered_tissue_dict = load_8cubed_filtered_data(adata_filtered_dir, obs_only=False, tissues_to_include=tissues_to_include)
adata_cellbender_dict = {}
for tissue in adata_filtered_tissue_dict:
    print(f"Processing CellBender data from tissue {tissue}...")
    if adata_filtered_tissue_dict[tissue].var.index.name == "gene_id":  # make sure gene_name is index, gene_id is a column
        adata_filtered_tissue_dict[tissue].var["gene_id"] = adata_filtered_tissue_dict[tissue].var.index
        adata_filtered_tissue_dict[tissue].var = adata_filtered_tissue_dict[tissue].var.set_index("gene_name")
    # Map predicted_doublet to boolean (had some "True"/"False" strings)
    adata_filtered_tissue_dict[tissue].obs["predicted_doublet"] = adata_filtered_tissue_dict[tissue].obs["predicted_doublet"].astype(str).fillna("False").map({"True": True, "False": False}).astype(bool)
    for plate in adata_filtered_tissue_dict[tissue].obs["plate"].unique().tolist():
        if plate not in plates:
            continue
        if plate not in adata_cellbender_dict:
            adata_cellbender_dict[plate] = adata_filtered_tissue_dict[tissue][adata_filtered_tissue_dict[tissue].obs["plate"] == plate].copy()
        else:
            adata_cellbender_dict[plate] = adata_cellbender_dict[plate].concatenate(adata_filtered_tissue_dict[tissue][adata_filtered_tissue_dict[tissue].obs["plate"] == plate])

# for plate, adata in adata_cellbender_dict.items():
#     adata_path = os.path.join(data_dir, plate, "cellbender.h5ad")
#     if not os.path.exists(adata_path):
#         os.makedirs(os.path.dirname(adata_path), exist_ok=True)
#         adata.write_h5ad(adata_path)

## Merge cellbender clusters into raw adatas

In [ ]:
for plate in adata_raw_dict.keys():
    adata_raw = adata_raw_dict[plate]
    if "leiden" in adata_raw.obs.columns:
        print(f"[WARN] Plate {plate} already has leiden column — skipping")
        continue
    adata_cb = adata_cellbender_dict[plate]

    adata_raw.obs = adata_raw.obs.join(
        adata_cb.obs[["leiden"]],
        how="left",
        rsuffix="_cellbender",
    )

    adata_raw_dict[plate] = adata_raw

## Identify and filter empty droplets

In [ ]:
adata_raw_filtered_dict = {}
for plate, adata_raw in adata_raw_dict.items():
    adata_raw = cs_utils.infer_empty_droplets(adata_raw, method="threshold", umi_cutoff=umi_cutoffs[plate], verbose=verbose)  # adds adata.obs["is_empty"]
    adata_raw.var['empty_counts'] = np.array(adata_raw.X[adata_raw.obs['is_empty'].values, :].sum(axis=0)).flatten()
    adata_raw_dict[plate] = adata_raw
    adata_raw_filtered = adata_raw[~adata_raw.obs["is_empty"]].copy()
    adata_raw_filtered_dict[plate] = adata_raw_filtered

In [ ]:
for plate in adata_raw_dict.keys():
    total_counts = adata_raw_dict[plate].X.sum()
    print(f"Total barcodes (cells + empty) in plate {plate}: {adata_raw_dict[plate].n_obs}, total expected cells: {expected_cells[plate]}, total genes: {adata_raw_dict[plate].n_vars}, total counts: {total_counts}")

In [ ]:
adata_raw_dict

In [ ]:
adata_raw_filtered_dict

## Save Anndatas

In [ ]:
for plate, adata in adata_raw_dict.items():
    adata_path = os.path.join(data_dir, plate, "raw_counts.h5ad")
    if not os.path.exists(adata_path):
        os.makedirs(os.path.dirname(adata_path), exist_ok=True)
        adata.write_h5ad(adata_path)

for plate, adata in adata_raw_filtered_dict.items():
    adata_path = os.path.join(data_dir, plate, "raw_counts_removed_empty_barcodes.h5ad")
    if not os.path.exists(adata_path):
        os.makedirs(os.path.dirname(adata_path), exist_ok=True)
        adata.write_h5ad(adata_path)

for plate, adata in adata_cellbender_dict.items():
    adata_path = os.path.join(data_dir, plate, "cellbender.h5ad")
    if not os.path.exists(adata_path):
        os.makedirs(os.path.dirname(adata_path), exist_ok=True)
        adata.write_h5ad(adata_path)
    if not os.path.exists(gene_id_name_map_path):
        # gene_name is in adata.var and doesn't start with ENSMUSG, and gene_id is in adata.var and starts with ENSMUSG
        if adata.var.index.name == "gene_id":
            adata.var["gene_id"] = adata.var.index
        if adata.var.index.name == "gene_name":
            adata.var["gene_name"] = adata.var.index
        if "gene_name" not in adata.var.columns or "gene_id" not in adata.var.columns or adata.var["gene_name"].str.startswith("ENSMUSG").all() or not adata.var["gene_id"].str.startswith("ENSMUSG").all():
            continue
        # no duplicates AND default index
        gene_map_df = pd.DataFrame({
            "gene_id": adata.var["gene_id"],
            "gene_name": adata.var["gene_name"]
        }).drop_duplicates().reset_index(drop=True)
        gene_map_df.loc[len(gene_map_df)] = ["ENSMUSG00000051747.17", "Ttn"]  # add missing Ttn gene
        gene_map_df.to_csv(gene_id_name_map_path, index=False)

## Markers

In [ ]:
if not os.path.exists(eight_cubed_markers_path):
    eight_cubed_markers_url = "https://docs.google.com/spreadsheets/d/1RJHnxeobFfXAQdrUGAn4SWxQZzs0KaheuLJgNc6AjCg/export?format=csv&gid=0"
    r = requests.get(eight_cubed_markers_url)
    with open(eight_cubed_markers_path, "wb") as f:
        f.write(r.content)

eight_cubed_markers_df = pd.read_csv(eight_cubed_markers_path)
if "gene_id" not in eight_cubed_markers_df.columns:
    # make gene_name_to_symbol_df
    files = sorted([os.path.join(adata_filtered_dir, f) for f in os.listdir(adata_filtered_dir) if f.endswith(".h5ad")])
    first_file = files[0]
    adata_tmp = ad.read_h5ad(first_file, backed="r")
    gene_name_to_symbol_df = adata_tmp.var[["gene_name", "gene_id"]].copy()
    gene_name_to_symbol_df = gene_name_to_symbol_df.reset_index(drop=True)
    eight_cubed_markers_df = eight_cubed_markers_df.merge(gene_name_to_symbol_df, left_on="ai", right_on="gene_name", how="left")
    eight_cubed_markers_df.to_csv(eight_cubed_markers_path, index=False)

## cellsweep

### Run this or scripts/run_8cube.py

In [ ]:
max_ram_gb = 500  # 300 GB

# Set max RAM usage in bytes
if max_ram_gb is None:
    max_ram_gb = 16

MAX_RAM = max_ram_gb * 1024**3

# plates = ["igvf_003", "igvf_004", "igvf_005", "igvf_007", "igvf_008b", "igvf_009", "igvf_010", "igvf_011"]
# adata_raw_dict = {}
# for plate in plates:
#     adata_raw_dict[plate] = ad.read_h5ad(os.path.join(data_dir, plate, "raw_counts.h5ad"))

# expected_cells = {
#     'igvf_003': 643226,
#     'igvf_004': 679838,
#     'igvf_005': 722870,
#     'igvf_007': 879650,
#     'igvf_008b': 606911,
#     'igvf_009': 772261,
#     'igvf_010': 844946,
#     'igvf_011': 806290
# }

adata_cellsweep_dict = {}
# for plate, adata_raw in adata_raw_dict.items():
try:
    for plate in plates:
        adata_path_cellsweep = os.path.join(data_dir, plate, "cellsweep.h5ad")
        if os.path.exists(adata_path_cellsweep) and not overwrite:
            print(f"Cellsweep output for plate {plate} already exists at {adata_path_cellsweep}, skipping...")
            continue
        print(f"Processing Cellsweep for plate {plate}...")
        adata_raw = ad.read_h5ad(os.path.join(data_dir, plate, "raw_counts.h5ad"))
        cellsweep_log_path = os.path.join(data_dir, plate, "cellsweep.log")
        
        adata_cellsweep = denoise_count_matrix(adata_raw, adata_out=adata_path_cellsweep, beta=cellsweep_init_beta, freeze_ambient_profile=True, init_alpha=cellsweep_init_alpha, max_iter=cellsweep_max_iter, empty_droplet_method="threshold", expected_cells=expected_cells[plate], threads=threads, verbose=verbose, log_file=cellsweep_log_path)
        # adata_cellsweep = adata_cellsweep[~adata_cellsweep.obs["is_empty"]].copy()
        # adata_cellsweep.var_names_make_unique()
        # adata_filtered_path_cellsweep = os.path.join(data_dir, plate, "cellsweep_filtered.h5ad")
        # if not os.path.exists(adata_filtered_path_cellsweep) or overwrite:
        #     adata_cellsweep.write_h5ad(adata_filtered_path_cellsweep)
        # adata_cellsweep_dict[plate] = adata_cellsweep

        adata_cellsweep = None  #? memory management
        del adata_raw   #? memory management
except MemoryError:
    print("❌ Memory limit exceeded — exiting")  # might just print 'Segmentation fault (core dumped)' rather than this
    sys.exit(1)

# for plate, adata in adata_cellsweep_dict.items():
#     adata_path = os.path.join(data_dir, plate, "cellsweep.h5ad")
#     if not os.path.exists(adata_path):
#         os.makedirs(os.path.dirname(adata_path), exist_ok=True)
#         adata.write_h5ad(adata_path)

# Analysis

In [ ]:
for plate in adata_raw_dict.keys():
    print(f"Plate: {plate}")
    # print(f"Adata raw (all barcodes): {adata_raw_dict[plate]}")
    # print(f"Adata raw (filtered barcodes): {adata_raw_filtered_dict[plate]}")
    print(f"Adata cellsweep: {adata_cellsweep_dict[plate]}\n\n")

## 8 Cubed Analysis

### Run this or scripts/visualize_8cube.py

In [ ]:
import os
import anndata as ad
import pandas as pd
import numpy as np
import itertools
import resource
import sys
import cellsweep.utils as cs_utils

debug = False
plates = ["igvf_003", "igvf_004", "igvf_005", "igvf_007", "igvf_008b", "igvf_009", "igvf_010", "igvf_011"]  # ["igvf_003"]  #? debug
include_cellbender = False
print_custom_markers = True
overwrite = False
max_ram_gb = 600  # GB
custom_markers = {
    'CortexHippocampus': ["Snap25", "Nrxn3", "Nrxn1"],  # Snap25: found in plate 3, tissue heart, cluster 36; Nrxn3, Nrxn1: found in plate 11, tissue gastroc, cluster 38
    'Heart': [],
    'Liver': ["Alb"],  # found in plate 5, tissue heart, cluster 30
    'HypothalamusPituitary': [],
    'Gonads': [],
    'Adrenal': ["Star"],  # found in plate 9, tissue kidney, cluster 14
    'Kidney': ["Slc34a1"],  # found in plate 10, tissue gastroc, cluster 32
    'Gastrocnemius': ["Myh4"]  # found in plate 10, tissue kidney, clusters 0,28,29,30
}
# custom_markers = {
#     'CortexHippocampus': ["Nrxn3", "Nrxn1", "Meis2", "Slc17a7", "Mir124a-1hg", "Snap25"],
#     'Heart': [],
#     'Liver': ["Cyp1a2", "Ttr", "Alb"],
#     'HypothalamusPituitary': [],
#     'Gonads': [],
#     'Adrenal': ["Chga", "Star"],
#     'Kidney': ["Slc5a2", "Slc34a1", "Akr1c21"],
#     'Gastrocnemius': ["Myh4", "Myh2", "Myh1"]
# }

# cellsweep_dir = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
# data_dir = os.path.join(cellsweep_dir, "notebooks", "data2", "8cubed")
# eight_cubed_markers_path = os.path.join(data_dir, "8_cube_marker_genes.csv")
# gene_id_name_map_path = os.path.join(data_dir, "gene_id_name_map.csv")
# out_dir = os.path.join(cellsweep_dir, "notebooks", "output", "8cubed")
# os.makedirs(out_dir, exist_ok=True)

# Set max RAM usage in bytes
if max_ram_gb is None:
    max_ram_gb = 16

MAX_RAM = max_ram_gb * 1024**3

soft, hard = resource.getrlimit(resource.RLIMIT_AS)
resource.setrlimit(resource.RLIMIT_AS, (MAX_RAM, MAX_RAM))

all_custom_markers_start_with_ensmug = all(gene.startswith("ENSMUG") for genes in custom_markers.values() for gene in genes)
gene_name_to_id = None

try:
    adata_raw_filtered_dict, adata_cellsweep_dict, adata_cellbender_dict = {}, {}, {}
    for plate in plates:
        print(f"Loading data for plate {plate}...")
        adata_raw_filtered_path = os.path.join(data_dir, plate, "raw_counts_removed_empty_barcodes.h5ad")
        if not os.path.exists(adata_raw_filtered_path):
            print(f"  File {adata_raw_filtered_path} does not exist, skipping...")
            continue
        adata_raw_filtered = ad.read_h5ad(adata_raw_filtered_path)
        adata_raw_filtered.var_names_make_unique()
        if debug:  # filter to 5000 cells for debugging
            np.random.seed(42)
            adata_raw_filtered = adata_raw_filtered[np.random.choice(adata_raw_filtered.n_obs, size=5000, replace=False), :].copy()
            barcodes = adata_raw_filtered.obs_names
        adata_raw_filtered_dict[plate] = adata_raw_filtered
        adata_raw_filtered = None  # free memory

        adata_cellsweep_path = os.path.join(data_dir, plate, "cellsweep.h5ad")
        if not os.path.exists(adata_cellsweep_path):
            print(f"  File {adata_cellsweep_path} does not exist, skipping...")
            continue
        adata_cellsweep = ad.read_h5ad(adata_cellsweep_path)
        adata_cellsweep = adata_cellsweep[~adata_cellsweep.obs["is_empty"]].copy()
        adata_cellsweep.var_names_make_unique()
        if debug:  # filter to the same 5000 cells as above for debugging
            adata_cellsweep = adata_cellsweep[adata_cellsweep.obs_names.isin(barcodes), :].copy()
        adata_cellsweep_dict[plate] = adata_cellsweep
        adata_cellsweep = None  # free memory

        if not include_cellbender and all_custom_markers_start_with_ensmug:
            continue
        
        adata_cellbender_path = os.path.join(data_dir, plate, "cellbender.h5ad")
        if not os.path.exists(adata_cellbender_path):
            print(f"  File {adata_cellbender_path} does not exist, skipping...")
            continue
        adata_cellbender = ad.read_h5ad(adata_cellbender_path)
        
        if custom_markers is not None and len(custom_markers) > 0 and gene_name_to_id is None:
            if "gene_name" not in adata_cellbender.var.columns or "gene_id" not in adata_cellbender.var.columns or adata_cellbender.var["gene_name"].str.startswith("ENSMUSG").all() or not adata_cellbender.var["gene_id"].str.startswith("ENSMUSG").all():
                gene_name_to_id = pd.read_csv(gene_id_name_map_path).set_index("gene_name")["gene_id"].to_dict()
            else:    
                gene_name_to_id = adata_cellbender.var.set_index("gene_name")["gene_id"].to_dict()
        
        if not all_custom_markers_start_with_ensmug:
            for tissue in custom_markers:
                gene_ids = []
                for gene_name in custom_markers[tissue]:
                    gene_id = gene_name_to_id.get(gene_name)
                    if gene_id is not None:
                        gene_ids.append(gene_id)
                custom_markers[tissue] = gene_ids
            all_custom_markers_start_with_ensmug = True
        if not include_cellbender:
            adata_cellbender = None  # free memory
            continue

        if adata_cellbender.obs_names[0].endswith("-0") or adata_cellbender.obs_names[0].endswith("-1"):  # strip "-0/-1" suffix from barcodes added by CellBender
            adata_cellbender.obs_names = [bc[:-2] for bc in adata_cellbender.obs_names]
        if not adata_cellbender.var_names[0].startswith("ENSMUSG"):
            adata_cellbender.var_names = adata_cellbender.var["gene_id"].astype(str)  # Assign gene_id as the new index
        adata_cellbender.var_names_make_unique()
        if debug:  # filter to the same 5000 cells as above for debugging
            adata_cellbender = adata_cellbender[adata_cellbender.obs_names.isin(barcodes), :].copy()
        adata_cellbender_dict[plate] = adata_cellbender
        adata_cellbender = None  # free memory
    
    if custom_markers is not None and len(custom_markers) > 0 and not all_custom_markers_start_with_ensmug:
        raise ValueError("Custom markers contain gene names that were not found in CellBender data; cannot proceed.")

    # np.random.seed(42)
    # adata_cellsweep_dict["igvf_003"] = adata_cellsweep_dict["igvf_003"][np.random.choice(adata_cellsweep_dict["igvf_003"].n_obs, size=5000, replace=False), :].copy()  #? debug

    dict_of_adata_dicts = {
        "raw": adata_raw_filtered_dict,
        "cellsweep": adata_cellsweep_dict,
        "cellbender": adata_cellbender_dict,
    }
    if not include_cellbender:
        dict_of_adata_dicts.pop("cellbender")
    print("Generating 8cubed plots...")
    cs_utils.make_8cubed_plots(dict_of_adata_dicts, eight_cubed_markers_path, custom_markers=custom_markers, gene_name_to_id=gene_name_to_id, print_custom_markers=print_custom_markers, out_dir=out_dir, overwrite=overwrite)
except MemoryError:
    print("❌ Memory limit exceeded — exiting")  # might just print 'Segmentation fault (core dumped)' rather than this
    sys.exit(1)


In [ ]:
import anndata as ad
import os
import pandas as pd
import cellsweep.utils as cs_utils
cellsweep_dir = os.path.dirname(os.path.abspath(""))
data_dir = os.path.join(cellsweep_dir, "notebooks", "data", "8cubed")
out_dir = os.path.join(cellsweep_dir, "notebooks", "output", "8cubed")
eight_cubed_markers_path = os.path.join(data_dir, "8_cube_marker_genes.csv")

eight_cubed_markers_df = pd.read_csv(eight_cubed_markers_path, usecols=["gene_id", "gene_name", "Tissue"])
eight_cubed_markers_df["Tissue"] = eight_cubed_markers_df["Tissue"].replace({
    "GonadsMale": "Gonads",
    "GonadsFemale": "Gonads",
})

plates = ["igvf_003", "igvf_004", "igvf_005", "igvf_007", "igvf_008b", "igvf_009", "igvf_010", "igvf_011"]
for plate in plates:
    print(f"Plate {plate}")
    adata_cellsweep = ad.read_h5ad(os.path.join(data_dir, plate, "cellsweep.h5ad"), backed="r")
    # adata_cellsweep = adata_cellsweep[~adata_cellsweep.obs["is_empty"]].copy()
    adata_cellsweep.var_names_make_unique()

    # plot ambient genes predicted
    cs_utils.plot_empty_gene_counts(adata_cellsweep, title=f"Counts per gene in empty droplets - Plate {plate}", highlight_indices=None, out_path=os.path.join(out_dir, f"empty_gene_counts_plate_{plate}.png"), show=True)

    # look at the "true" ambient genes (ie the marker genes for the incorrect tissues)
    tissues = adata_cellsweep.obs["Tissue"].unique().tolist()
    if len(tissues) != 2:
        print(f"Warning: Plate {plate} does not have exactly 2 tissues in adata_cellsweep. Found tissues: {tissues}. Skipping.")
        continue

    eight_cubed_markers_df_filtered = eight_cubed_markers_df[eight_cubed_markers_df["Tissue"].isin(tissues)]

    adata_cellsweep.var["ambient_rank"] = adata_cellsweep.var["ambient_hat"].rank(ascending=False, method="dense")

    ambient_df = adata_cellsweep.var[["ambient_hat", "ambient_rank"]].copy()
    eight_cubed_markers_df_filtered = eight_cubed_markers_df_filtered.merge(ambient_df, on="gene_id", how="left")
    eight_cubed_markers_df_filtered["ambient_rank"] = eight_cubed_markers_df_filtered["ambient_rank"].astype("Int64")

    eight_cubed_markers_df_filtered = eight_cubed_markers_df_filtered.sort_values(by="ambient_rank", ascending=True)
    eight_cubed_markers_df_filtered = eight_cubed_markers_df_filtered[~eight_cubed_markers_df_filtered["ambient_rank"].isna()]

    # plot ambient genes predicted
    cs_utils.plot_empty_gene_counts(adata_cellsweep, title=f"Counts per gene in empty droplets - Plate {plate}", highlight_indices=eight_cubed_markers_df_filtered["ambient_rank"], out_path=os.path.join(out_dir, f"empty_gene_counts_plate_{plate}_with_overlay.png"), show=True)
    
    # print eight_cubed_markers_df_filtered sorted ascending by ambient_rank
    with pd.option_context(
        "display.max_rows", 500,
        "display.max_columns", None,
        "display.width", 10_000,       # huge width prevents line breaks
        "display.max_colwidth", None  # prevents truncation
    ):
        print(eight_cubed_markers_df_filtered.head(500))
    
    print("--------------------------------\n")

    # break

## Trem2

In [ ]:
adata_raw_trem2_dir = "/mnt/data1/8_cube_trem2_raw"
trem2_metadata_path = "/mnt/data1/igvf015_obs.csv.gz"

In [ ]:
# !cd {adata_raw_trem2_dir} && curl -L -o subpool_67C.h5ad "https://api.data.igvf.org/matrix-files/IGVFFI7692WGUI/@@download/IGVFFI7692WGUI.h5ad" && curl -L -o subpool_67D.h5ad "https://api.data.igvf.org/matrix-files/IGVFFI1306QBMO/@@download/IGVFFI1306QBMO.h5ad" && curl -L -o subpool_67E.h5ad "https://api.data.igvf.org/matrix-files/IGVFFI6709CSRA/@@download/IGVFFI6709CSRA.h5ad" && curl -L -o subpool_67F.h5ad "https://api.data.igvf.org/matrix-files/IGVFFI7088RATR/@@download/IGVFFI7088RATR.h5ad"

In [ ]:
trem2_metadata_df = pd.read_csv(trem2_metadata_path, compression="gzip")
trem2_metadata_df = trem2_metadata_df.rename(columns={"Celltype": "celltype"})
gene_id_to_name = pd.read_csv(gene_id_name_map_path).set_index("gene_id")["gene_name"].to_dict()

### Load in

In [ ]:
adata_raw_trem2_list = []
files = sorted([os.path.join(adata_raw_trem2_dir, f) for f in os.listdir(adata_raw_trem2_dir) if f.endswith(".h5ad")])
for filename in files:
    print(f"Loading {filename}...")
    subpool = os.path.basename(filename).replace(".h5ad", "").replace("sublibrary_", "")
    adata = ad.read_h5ad(filename, backed=None)
    adata.obs["subpool"] = subpool
    adata_raw_trem2_list.append(adata)
# concatenate
adata_raw_trem2 = ad.concat(adata_raw_trem2_list, join="outer")
adata_raw_trem2.obs[["barcode", "sample"]] = adata_raw_trem2.obs.index.to_series().str.rsplit("_", n=1, expand=True)
adata_raw_trem2.var["gene_name"] = adata_raw_trem2.var_names.map(gene_id_to_name)
# adata_raw_trem2.obs["subpool"] = adata_raw_trem2.obs["subpool"].astype("category")  # gives error
adata_raw_trem2.obs.index.name = "barcode_sample"  # rename adata_raw_trem2 index_name to "barcode_sample"
adata_raw_trem2

### Merge in tissue data

In [ ]:
adata_raw_trem2.obs = adata_raw_trem2.obs.merge(
    trem2_metadata_df[["barcode", "subpool", "Tissue", "celltype", "leiden"]],
    on=["barcode", "subpool"],
    how="left"
)

adata_raw_trem2.obs["tissue_specific_celltype"] = adata_raw_trem2.obs["celltype"]
adata_raw_trem2.obs["celltype"] = adata_raw_trem2.obs["Tissue"] + "_" + adata_raw_trem2.obs["celltype"].fillna("empty")

In [ ]:
adata_raw_trem2.obs["Tissue"].value_counts()

### Knee plot and empty inference

In [ ]:
_ = cs_utils.knee_plot(adata_raw_trem2)

In [ ]:
# (1) infer empty barcodes from UMI threshold
# umi_cutoff = 300  #!! set from knee plot
# adata_raw_trem2 = cs_utils.infer_empty_droplets(adata_raw_trem2, method="threshold", umi_cutoff=umi_cutoff, verbose=verbose)  # adds adata.obs["is_empty"]

# (2) infer empty barcodes from celltype being NA (pre-annotated)
adata_raw_trem2.obs['is_empty'] = adata_raw_trem2.obs['celltype'].isna()
adata_raw_trem2.var['empty_counts'] = np.array(adata_raw_trem2.X[adata_raw_trem2.obs['is_empty'].values, :].sum(axis=0)).flatten()
adata_raw_trem2.obs.loc[~adata_raw_trem2.obs["is_empty"]].shape[0]

### save

In [ ]:
adata_path = os.path.join(data_dir, "trem2", "raw_counts.h5ad")
if not os.path.exists(adata_path):
    os.makedirs(os.path.dirname(adata_path), exist_ok=True)
    adata_raw_trem2.write_h5ad(adata_path)

### Run cellsweep

In [ ]:
# adata_path = os.path.join(data_dir, "trem2", "raw_counts.h5ad")
# adata_raw_trem2 = ad.read_h5ad(adata_path)
# overwrite = True

adata_path_cellsweep_trem2 = os.path.join(data_dir, "trem2", "cellsweep.h5ad")
if not os.path.exists(adata_path_cellsweep_trem2) or overwrite:
    print(f"Processing Cellsweep trem2")
    # adata_raw = ad.read_h5ad(os.path.join(data_dir, plate, "raw_counts.h5ad"))
    cellsweep_log_path = os.path.join(data_dir, "trem2", "cellsweep.log")
    adata_cellsweep_trem2 = denoise_count_matrix(adata_raw_trem2, adata_out=adata_path_cellsweep_trem2, beta=cellsweep_init_beta, freeze_ambient_profile=True, init_alpha=cellsweep_init_alpha, max_iter=cellsweep_max_iter, empty_droplet_method="threshold", threads=threads, verbose=verbose, log_file=cellsweep_log_path)
else:
    print(f"Cellsweep output for trem2 already exists at {adata_path_cellsweep_trem2}, skipping...")
    adata_cellsweep_trem2 = ad.read_h5ad(adata_path_cellsweep_trem2)

In [ ]:
import matplotlib.pyplot as plt
print(adata_cellsweep_trem2.obs.loc[~adata_cellsweep_trem2.obs["is_empty"]].shape)
plt.figure(figsize=(8,6))
plt.hist(adata_cellsweep_trem2.obs.loc[~adata_cellsweep_trem2.obs["is_empty"], "alpha_hat"], bins=100, color='blue', alpha=0.7)
plt.title("Histogram of alpha_hat in Cellsweep trem2")
plt.xlabel("alpha_hat")
plt.ylabel("Number of Cells")
plt.grid(axis='y', alpha=0.75)
plt.show()

In [ ]:
adata_raw_trem2 = ad.read_h5ad(adata_path)
adata_cellsweep_trem2 = ad.read_h5ad(adata_path_cellsweep_trem2)

# remove empty
adata_raw_trem2 = adata_raw_trem2[~adata_raw_trem2.obs["is_empty"]].copy()
adata_cellsweep_trem2 = adata_cellsweep_trem2[~adata_cellsweep_trem2.obs["is_empty"]].copy()

In [ ]:
# normalization and log1p
# import scanpy as sc
# sc.pp.normalize_total(adata_raw_trem2, target_sum=1e4)
# sc.pp.log1p(adata_raw_trem2)

# sc.pp.normalize_total(adata_cellsweep_trem2, target_sum=1e4)
# sc.pp.log1p(adata_cellsweep_trem2)

In [ ]:
# for debugging
# np.random.seed(42)
# adata_raw_trem2.obs["Tissue"] = np.random.choice(["Liver", "Heart"], size=adata_raw_trem2.n_obs)
# adata_cellsweep_trem2 = adata_raw_trem2.copy()

custom_markers = {
    # 'CortexHippocampus': ["Snap25", "Nrxn3", "Nrxn1"],
    # 'Heart': [],
    'Liver': ["Alb"],
    # 'HypothalamusPituitary': [],
    # 'Gonads': [],
    # 'Adrenal': ["Star"],
    # 'Kidney': ["Slc34a1"],
    'Gastrocnemius': ["Myh4", "Ttn", "Neb", "Pde4d", "Akap6"]
}

log_y = True

# keep only genes that exist
for gene_tissue, genes in custom_markers.items():
    for gene in genes:
        series_list = []
        for adata_name, adata_object in {"raw": adata_raw_trem2, "cellsweep": adata_cellsweep_trem2}.items():
            if adata_object is None:
                continue

            if gene not in adata_object.var["gene_name"].values:
                print(f"Warning: Gene {gene} not found in adata object var.")
                continue
            
            gi = adata_object.var["gene_name"].astype(str).tolist().index(gene)
            
            counts = {}
            tissues = adata_object.obs["Tissue"].dropna().unique().tolist()
            
            for tissue in tissues:
                mask = adata_object.obs["Tissue"] == tissue
                X_sub = adata_object[mask, gi].X  # column slice → (cells × 1)

                total = X_sub.sum() if sp.issparse(X_sub) else np.sum(X_sub)  # sum over all cells for this tissue
                total = total / mask.sum()  # take the mean per cell
                # total = max(0, math.log2(total + 1))  # take the log2p (matches paper)
                counts[tissue] = float(total)

            s = pd.Series(counts, name=adata_name)
            series_list.append(s)

        if len(series_list) == 0:
            continue
        
        df = pd.concat(series_list, axis=1).fillna(0)
        if log_y:
            df = df + 1e-6  # avoid log(0)

        plt.figure(figsize=(10, 4))
        df.plot(kind="bar", width=0.8)
        plt.ylabel("average counts")
        plt.xlabel("Tissue")
        if log_y:
            plt.yscale("log")
        plt.title(f"{gene} (a {gene_tissue} marker) average counts per tissue in TREM2")
        plt.xticks(rotation=90, ha="right")
        plt.legend(title="Dataset")
        plt.tight_layout()

        out_path = os.path.join(out_dir, "trem2", f"{gene}_counts_per_tissue_trem2.png")
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        plt.savefig(out_path, dpi=300)

        plt.show()

In [ ]:
import os
import anndata as ad
import cellsweep.utils as cs_utils

plate = "igvf_009"

cellsweep_dir = os.path.dirname(os.path.abspath(""))
data_dir = os.path.join(cellsweep_dir, "notebooks", "data", "8cubed")
adata = ad.read_h5ad(f'{data_dir}/{plate}/cellsweep.h5ad', backed='r')
adata.obs["Tissue"].unique()

In [ ]:
tissue1 = "Adrenal"
tissue2 = "Kidney"

print(tissue1, adata.obs.loc[adata.obs["Tissue"] == tissue1, "celltype"].value_counts(), "\n------------\n")
print(tissue2, adata.obs.loc[adata.obs["Tissue"] == tissue2, "celltype"].value_counts())

In [ ]:
tissue1_celltype = "zona fasciculata"
tissue2_celltype = "proximal tubule epithelial cell"

outpath1 = f"{cellsweep_dir}/{tissue1}.png"
outpath2 = f"{cellsweep_dir}/{tissue2}.png"
cs_utils.make_dummy_title(f"Plate: {plate}\nTissue: {tissue1}\nCelltype: {tissue1_celltype}", outpath1)
cs_utils.make_dummy_title(f"Plate: {plate}\nTissue: {tissue2}\nCelltype: {tissue2_celltype}", outpath2)

In [ ]:
import os
import numpy as np
import pandas as pd
import anndata as ad
import cellsweep.utils as cs_utils

runs = [
    {
        "plate": "igvf_003",
        "tissue1": "Heart",
        "tissue2": "CortexHippocampus",
        "celltype1": "atrial cardiac myocyte",  # None to plot for all cell types aggregate
        "celltype2": "glutamatergic neuron",  # None to plot for all cell types aggregate
        "gene_name_to_marker_type": {
            "Snap25": "cortex/hippocampus"
        }
    },
    {
        "plate": "igvf_009",
        "tissue1": "Adrenal",
        "tissue2": "Kidney",
        "celltype1": "zona fasciculata",  # None to plot for all cell types aggregate
        "celltype2": "proximal tubule epithelial cell",  # None to plot for all cell types aggregate
        "gene_name_to_marker_type": {
            "Star": "adrenal"
        }
    },
]

In [ ]:
cellsweep_dir = os.path.dirname(os.path.abspath(""))
data_dir = os.path.join(cellsweep_dir, "notebooks", "data", "8cubed")
out_dir = os.path.join(cellsweep_dir, "notebooks", "output", "8cubed")
gene_id_name_map_path = os.path.join(data_dir, "gene_id_name_map.csv")
gene_df = pd.read_csv(gene_id_name_map_path)
gene_name_to_id_dict = dict(zip(gene_df["gene_name"], gene_df["gene_id"]))

In [ ]:
for run in runs:
    plate, tissue1, tissue2, celltype1, celltype2, gene_name_to_marker_type = run["plate"], run["tissue1"], run["tissue2"], run["celltype1"], run["celltype2"], run["gene_name_to_marker_type"]
    print(f"Processing plate {plate} for celltypes {celltype1} and {celltype2}...")
    
    adata_cellsweep_dir = f'{data_dir}/{plate}'
    adata = ad.read_h5ad(f'{adata_cellsweep_dir}/cellsweep.h5ad')
    out_dir_specific = os.path.join(out_dir, f"plate_{plate}", "scatterplots_new")
    os.makedirs(out_dir_specific, exist_ok=True)
    
    cs_utils.make_8cube_scatterplot(adata=adata, celltype1=celltype1, celltype2=celltype2, tissue1=tissue1, tissue2=tissue2, gene_name_to_marker_type=gene_name_to_marker_type, gene_name_to_id_dict=gene_name_to_id_dict, out_dir=out_dir_specific, overwrite=True)
    cs_utils.make_dummy_title(f"Plate: {plate}\nTissue: {tissue1}\nCelltype: {celltype1}", os.path.join(out_dir_specific, f"{celltype1.replace(' ', '_')}_title.png"))
    cs_utils.make_dummy_title(f"Plate: {plate}\nTissue: {tissue2}\nCelltype: {celltype2}", os.path.join(out_dir_specific, f"{celltype2.replace(' ', '_')}_title.png"))

In [ ]:
# adata = ad.read_h5ad(f'{data_dir}/{plate}/cellsweep.h5ad')
# adata_zona_fasciculata = adata[adata.obs["celltype"] == "zona fasciculata"].copy()
# adata_proximal_tubule = adata[adata.obs["celltype"] == "proximal tubule epithelial cell"].copy()

# gene_id_name_map_path = os.path.join(data_dir, "gene_id_name_map.csv")
# gene_df = pd.read_csv(gene_id_name_map_path)

In [ ]:
# adata_zona_fasciculata.var["total_counts_cellsweep"] = np.asarray(adata_zona_fasciculata.X.sum(axis=0)).ravel()
# adata_zona_fasciculata.var["total_counts_raw"] = np.asarray(adata_zona_fasciculata.layers["raw"].sum(axis=0)).ravel()

# adata_proximal_tubule.var["total_counts_cellsweep"] = np.asarray(adata_proximal_tubule.X.sum(axis=0)).ravel()
# adata_proximal_tubule.var["total_counts_raw"] = np.asarray(adata_proximal_tubule.layers["raw"].sum(axis=0)).ravel()

In [ ]:
# gene_name = 'Star'
# gene_id = gene_df.loc[gene_df['gene_name'] == gene_name].values[0][0]

# adata_zona_fasciculata.var.loc[adata_zona_fasciculata.var.index == gene_id]

In [ ]:
# adata_proximal_tubule.var.loc[adata_proximal_tubule.var.index == gene_id]

In [ ]:
# def make_gene_to_scatter_location_dict(adata, custom_markers):
#     gene_to_scatter_location_dict = {}
#     if custom_markers is not None:
#         for marker_tissue, marker_genes in custom_markers.items():
#             for marker in marker_genes:
#                 if marker in adata.var_names:
#                     x = adata.var.loc[marker, "total_counts_cellsweep"]
#                     y = adata.var.loc[marker, "total_counts_raw"]

#                     # match plotting logic exactly
#                     x = x if x > 0.5 else 0.5
#                     y = y if y > 0.5 else 0.5

#                     # x = np.log2(x)
#                     # y = np.log2(y)

#                     gene_to_scatter_location_dict[marker] = (x, y)
#     return gene_to_scatter_location_dict

In [ ]:
# out_path = None
# custom_markers = {"Adrenal": [gene_id]}
# gene_to_scatter_location_dict = make_gene_to_scatter_location_dict(adata_proximal_tubule, custom_markers)  # None
# cs_utils.plot_matrix_scatterplot(adata1=adata_proximal_tubule.var["total_counts_cellsweep"], adata2=adata_proximal_tubule.var["total_counts_raw"], cmap="Blues", label_to_scatter_location_dict=gene_to_scatter_location_dict, scale="log", point_type="custom", title="Gene counts", density_type="scatter_with_kde", x_axis="cellsweep", y_axis='raw', out_path=out_path, show=True)